# IMDB Reviews — Text Sentiment Classification
Preprocess the reviews, extract TF-IDF features, train Logistic Regression (with Multinomial Naive Bayes as a baseline), report Accuracy/F1, and show the top 10 predictive words per class.

In [1]:
import re
import numpy as np
import pandas as pd
from functools import lru_cache
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

df = pd.read_csv('IMDB_Dataset.csv')
print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 1. Quick checks

In [2]:
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\n', df['sentiment'].value_counts())

review       0
sentiment    0
dtype: int64

Duplicates: 418

 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
df = df.drop_duplicates().reset_index(drop=True)
df['label'] = (df['sentiment'] == 'positive').astype(int)   # 1 = positive, 0 = negative
print(df.shape)

(49582, 3)


## 2. Text preprocessing
1. Strip HTML tags (reviews contain `<br />`)
2. Lowercase
3. Remove punctuation, digits and other non-letters
4. Tokenize (split on whitespace)
5. Remove stopwords — negations (*not, no, nor, never…*) are **kept** because they flip sentiment
6. Stem with the Porter stemmer

In [4]:
negations = {'not', 'no', 'nor', 'never', 'none', 'nothing', 'nobody', 'nowhere', 'neither', 'cannot'}
stop_words = set(ENGLISH_STOP_WORDS) - negations

stemmer = PorterStemmer()
stem = lru_cache(maxsize=None)(stemmer.stem)   # cache: same word is stemmed only once

def preprocess(text):
    text = re.sub(r'<.*?>', ' ', text)          # 1. HTML tags
    text = text.lower()                          # 2. lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)        # 3. punctuation / digits
    tokens = text.split()                        # 4. tokenize
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]   # 5. stopwords
    return ' '.join(stem(t) for t in tokens)     # 6. stem

print('Original :', df['review'][1][:300])
print('\nCleaned  :', preprocess(df['review'][1])[:300])

Original : A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" 

Cleaned  : wonder littl product film techniqu unassum old time bbc fashion give comfort discomfort sens realism entir piec actor extrem chosen michael sheen not got polari voic pat truli seamless edit guid refer william diari entri not worth watch terrificli written perform piec master product great master com


In [5]:
df['clean_review'] = df['review'].apply(preprocess)
df[['clean_review', 'sentiment']].head()

,clean_review,sentiment
0,review mention watch just oz episod ll hook ri...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic famili littl boy jake think zombi closet...,negative
4,petter mattei love time money visual stun film...,positive


## 3. Train/test split and TF-IDF features
Unigrams + bigrams so phrases like *not good* become features; capped at the 50,000 most frequent terms. The vectorizer is fit on the training set only.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, min_df=5, max_df=0.9)
X_train_tfidf = tfidf.fit_transform(X_train)   # fit on train only
X_test_tfidf = tfidf.transform(X_test)

print('Train matrix:', X_train_tfidf.shape)
print('Test matrix :', X_test_tfidf.shape)

Train matrix: (39665, 50000)
Test matrix : (9917, 50000)


## 4. Train classifiers

In [7]:
lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

## 5. Accuracy and F1-score

In [8]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Multinomial Naive Bayes'],
    'Accuracy': [accuracy_score(y_test, lr_pred), accuracy_score(y_test, nb_pred)],
    'F1-score': [f1_score(y_test, lr_pred), f1_score(y_test, nb_pred)],
}).round(4)
results

,Model,Accuracy,F1-score
0,Logistic Regression,0.8960,0.8980
1,Multinomial Naive Bayes,0.8808,0.8821


In [9]:
print('Logistic Regression — classification report')
print(classification_report(y_test, lr_pred, target_names=['negative', 'positive'], digits=4))
print('Confusion matrix (rows = actual, cols = predicted):')
print(pd.DataFrame(confusion_matrix(y_test, lr_pred),
                   index=['actual neg', 'actual pos'], columns=['pred neg', 'pred pos']))

Logistic Regression — classification report
              precision    recall  f1-score   support

    negative     0.9081    0.8804    0.8940      4940
    positive     0.8848    0.9116    0.8980      4977

    accuracy                         0.8960      9917
   macro avg     0.8964    0.8960    0.8960      9917
weighted avg     0.8964    0.8960    0.8960      9917

Confusion matrix (rows = actual, cols = predicted):
            pred neg  pred pos
actual neg      4349       591
actual pos       440      4537


## 6. Top 10 predictive words per class
From the Logistic Regression coefficients: the largest positive weights push toward **positive**, the most negative weights push toward **negative**. Words are shown as stems (e.g. *wast* = waste/wasted).

In [10]:
words = np.array(tfidf.get_feature_names_out())
coefs = lr.coef_[0]

top_pos = np.argsort(coefs)[::-1][:10]
top_neg = np.argsort(coefs)[:10]

top_words = pd.DataFrame({
    'Rank': range(1, 11),
    'Positive word': words[top_pos],
    'Pos coef': coefs[top_pos].round(3),
    'Negative word': words[top_neg],
    'Neg coef': coefs[top_neg].round(3),
})
top_words.style.hide(axis='index')

Rank,Positive word,Pos coef,Negative word,Neg coef
1,great,8.476000,worst,-9.944000
2,excel,7.525000,bad,-8.545000
3,love,5.914000,aw,-8.367000
4,best,5.880000,wast,-7.869000
5,enjoy,5.876000,bore,-7.382000
6,perfect,5.708000,poor,-6.473000
7,amaz,5.060000,terribl,-5.999000
8,favorit,4.981000,disappoint,-5.802000
9,fun,4.748000,noth,-5.761000
10,brilliant,4.640000,fail,-5.503000


## 7. Test on new reviews

In [11]:
samples = ['An absolutely wonderful film, the acting was brilliant and I loved every minute.',
           'Total waste of time. The plot was boring and the acting was awful.',
           'It was not good at all.']
pred = lr.predict(tfidf.transform([preprocess(s) for s in samples]))
for s, p in zip(samples, pred):
    print(('positive' if p else 'negative').ljust(9), '|', s)

positive  | An absolutely wonderful film, the acting was brilliant and I loved every minute.
negative  | Total waste of time. The plot was boring and the acting was awful.
negative  | It was not good at all.


## 8. Summary

| Model | Accuracy | F1-score |
|---|---|---|
| **Logistic Regression** | **0.896** | **0.898** |
| Multinomial Naive Bayes | 0.881 | 0.882 |

- **Logistic Regression is the better model**: about 1.5 points higher on both metrics, with balanced precision/recall across the two classes.
- **Positive words**: *great, excel(lent), love, best, enjoy, perfect, amaz(ing), favorit(e), fun, brilliant*.
- **Negative words**: *worst, bad, aw(ful), wast(e), bore(d/ing), poor, terribl(e), disappoint, noth(ing), fail*.
- Keeping negations and adding bigrams lets the model handle phrases like *not good* correctly.
- Both lists are clean sentiment adjectives, showing the model learns real opinion words rather than movie-specific names or topics.